# Séries temporelles et Transformers — RNN, LSTM, Transformer, GPT et BERT

**Objectifs :**
- générer une série temporelle `y = sin(x)` ;
- transformer une série en exemples « séquence de 10 valeurs → valeur suivante » ;
- créer, entraîner et tester un `RNN`, un `LSTM` et un `Transformer` ;
- comparer les trois architectures ;
- tester un modèle de type **GPT** (prédire la suite d’un texte) ;
- tester un modèle de type **BERT** (prédire un mot caché) ;
- résoudre les exercices de la deuxième partie (régression, MNIST, Iris).

## 0. Préparation

Le notebook fonctionne sur CPU. S’il existe un GPU CUDA, PyTorch pourra l’utiliser automatiquement.

Décommentez la cellule suivante pour installer les dépendances (une seule fois).

In [ ]:
# !pip install torch torchvision numpy pandas matplotlib scikit-learn transformers

In [ ]:
import math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch version:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())
print("Device utilisé:", device)

# Première partie — Modèles de séquences

## 1. Générer une série temporelle

Une **série temporelle** est une suite de valeurs ordonnées dans le temps. Nous générons :

\[
y = \sin(x), \quad x \in [0, 60]
\]

In [ ]:
x = torch.linspace(0, 60, 1200)
series = torch.sin(x)

print("Nombre de points :", len(series))
print("Min / Max :", series.min().item(), series.max().item())

plt.figure(figsize=(9,3.5))
plt.plot(x.numpy(), series.numpy())
plt.title("Série temporelle y = sin(x)")
plt.xlabel("x")
plt.ylabel("y")
plt.grid(True, alpha=0.3)
plt.show()

## 2. Créer les séquences d’apprentissage (fenêtre glissante)

On veut prédire la valeur suivante à partir des **10 valeurs précédentes** :

**[y_t, y_t+1, …, y_t+9] → y_t+10**

Les tenseurs auront les dimensions :
- `X` : `(N, 10, 1)` → (nombre d’exemples, longueur de séquence, nombre de variables) ;
- `y` : `(N, 1)`.

Pour une série temporelle, on **ne mélange pas** les données avant de séparer train/test : on garde les 80 % premiers points pour l’entraînement et les 20 % derniers pour le test.

In [ ]:
SEQ_LEN = 10

def create_sequences(series, seq_len=10):
    X, y = [], []
    for i in range(len(series) - seq_len):
        X.append(series[i:i + seq_len])
        y.append(series[i + seq_len])
    X = torch.stack(X).unsqueeze(-1)     # (N, seq_len, 1)
    y = torch.stack(y).unsqueeze(-1)     # (N, 1)
    return X, y

X, y = create_sequences(series, SEQ_LEN)
print("X :", X.shape, "| y :", y.shape)

split = int(0.8 * len(X))
X_train, y_train = X[:split], y[:split]
X_test, y_test = X[split:], y[split:]
print("Train :", X_train.shape, "| Test :", X_test.shape)

print("Exemple :", X_train[0].squeeze().numpy().round(3), "->", y_train[0].item())

In [ ]:
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)

# Découpage visuel train / test
plt.figure(figsize=(9,3.5))
plt.plot(x[:split + SEQ_LEN].numpy(), series[:split + SEQ_LEN].numpy(), label="Train")
plt.plot(x[split + SEQ_LEN:].numpy(), series[split + SEQ_LEN:].numpy(), label="Test")
plt.legend()
plt.title("Séparation train / test")
plt.grid(True, alpha=0.3)
plt.show()

## 3. Fonctions utilitaires : entraînement et test

Une seule fonction d’entraînement sert pour les trois modèles. C’est la boucle PyTorch habituelle :

**forward → loss → `zero_grad()` → `backward()` → `step()`**

In [ ]:
EPOCHS = 40

def count_params(model):
    return sum(p.numel() for p in model.parameters())


def train_model(model, epochs=EPOCHS, lr=1e-3):
    model.to(device)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    train_losses, test_losses = [], []
    for epoch in range(epochs):
        model.train()
        running = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)

            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()

            running += loss.item() * xb.size(0)

        train_losses.append(running / len(train_loader.dataset))

        model.eval()
        with torch.inference_mode():
            test_losses.append(criterion(model(X_test.to(device)), y_test.to(device)).item())

        if (epoch + 1) % 10 == 0:
            print(f"epoch={epoch + 1:3d} | train MSE={train_losses[-1]:.6f} | test MSE={test_losses[-1]:.6f}")
    return train_losses, test_losses


def evaluate(model):
    model.eval()
    with torch.inference_mode():
        pred = model(X_test.to(device)).cpu()
    mse = ((pred - y_test) ** 2).mean().item()
    mae = (pred - y_test).abs().mean().item()
    return pred, mse, mae


def plot_losses(train_losses, test_losses, title):
    plt.figure(figsize=(6,4))
    plt.plot(train_losses, label="Train")
    plt.plot(test_losses, label="Test")
    plt.yscale("log")
    plt.xlabel("Epoch")
    plt.ylabel("MSE")
    plt.title(title)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()


def plot_predictions(pred, title, n=150):
    plt.figure(figsize=(9,3.5))
    plt.plot(y_test[:n].numpy(), label="Réel")
    plt.plot(pred[:n].numpy(), "--", label="Prédit")
    plt.title(title)
    plt.xlabel("Pas de temps (jeu de test)")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

## 4. Le modèle RNN

Un **RNN** (*Recurrent Neural Network*) lit la séquence **pas à pas** et garde une mémoire, l’état caché `h_t` :

\[
h_t = \tanh(W_x x_t + W_h h_{t-1} + b)
\]

Nous utilisons le dernier état caché pour prédire la valeur suivante. Avec `batch_first=True`, l’entrée a la forme `(batch, temps, variables)`.

In [ ]:
class RNNModel(nn.Module):
    def __init__(self, input_size=1, hidden_size=32, num_layers=1):
        super().__init__()
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, h_n = self.rnn(x)          # out : (batch, temps, hidden)
        return self.fc(out[:, -1, :])   # dernier pas de temps -> valeur suivante

rnn_model = RNNModel()
print(rnn_model)
print("Paramètres :", count_params(rnn_model))

### Entraînement et test du RNN

In [ ]:
rnn_train, rnn_test = train_model(rnn_model)
plot_losses(rnn_train, rnn_test, "RNN — évolution du loss")

In [ ]:
pred_rnn, mse_rnn, mae_rnn = evaluate(rnn_model)
print(f"RNN  | MSE = {mse_rnn:.6f} | MAE = {mae_rnn:.6f}")
plot_predictions(pred_rnn, "RNN — prédictions sur le jeu de test")

## 5. Le modèle LSTM

Le **LSTM** (*Long Short-Term Memory*) ajoute à l’état caché une **cellule mémoire** `c_t` contrôlée par trois portes :
- porte d’**oubli** : que faut-il effacer de la mémoire ?
- porte d’**entrée** : que faut-il écrire ?
- porte de **sortie** : que faut-il exposer ?

Ces portes limitent le problème de disparition du gradient des RNN sur les longues séquences.

In [ ]:
class LSTMModel(nn.Module):
    def __init__(self, input_size=1, hidden_size=32, num_layers=1):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, (h_n, c_n) = self.lstm(x)  # un LSTM renvoie aussi la cellule mémoire c_n
        return self.fc(out[:, -1, :])

lstm_model = LSTMModel()
print(lstm_model)
print("Paramètres :", count_params(lstm_model))

### Entraînement et test du LSTM

In [ ]:
lstm_train, lstm_test = train_model(lstm_model)
plot_losses(lstm_train, lstm_test, "LSTM — évolution du loss")

In [ ]:
pred_lstm, mse_lstm, mae_lstm = evaluate(lstm_model)
print(f"LSTM | MSE = {mse_lstm:.6f} | MAE = {mae_lstm:.6f}")
plot_predictions(pred_lstm, "LSTM — prédictions sur le jeu de test")

## 6. Le modèle Transformer

Le **Transformer** ne lit pas la séquence pas à pas : grâce au mécanisme d’**attention**, chaque position regarde **toutes les autres en parallèle** :

\[
\text{Attention}(Q,K,V) = \text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)V
\]

Comme l’attention ne connaît pas l’ordre des éléments, on ajoute un **encodage positionnel** (sinusoïdal) :

\[
PE_{(pos,2i)} = \sin\left(\frac{pos}{10000^{2i/d}}\right), \qquad
PE_{(pos,2i+1)} = \cos\left(\frac{pos}{10000^{2i/d}}\right)
\]

Architecture utilisée : `Linear(1 → d_model)` → encodage positionnel → `TransformerEncoder` → dernière position → `Linear(d_model → 1)`.

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))    # (1, max_len, d_model)

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]


class TransformerModel(nn.Module):
    def __init__(self, d_model=32, nhead=4, num_layers=2, dim_ff=64, dropout=0.0):
        super().__init__()
        self.input_proj = nn.Linear(1, d_model)         # 1 valeur -> vecteur de dimension d_model
        self.pos_enc = PositionalEncoding(d_model)
        layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_ff,
            dropout=dropout, batch_first=True
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=num_layers, enable_nested_tensor=False)
        self.fc = nn.Linear(d_model, 1)

    def forward(self, x):
        z = self.pos_enc(self.input_proj(x))            # (batch, temps, d_model)
        z = self.encoder(z)                             # (batch, temps, d_model)
        return self.fc(z[:, -1, :])                     # dernière position -> valeur suivante

transformer_model = TransformerModel()
print(transformer_model)
print("Paramètres :", count_params(transformer_model))

### Entraînement et test du Transformer

In [ ]:
tr_train, tr_test = train_model(transformer_model)
plot_losses(tr_train, tr_test, "Transformer — évolution du loss")

In [ ]:
pred_tr, mse_tr, mae_tr = evaluate(transformer_model)
print(f"Transformer | MSE = {mse_tr:.6f} | MAE = {mae_tr:.6f}")
plot_predictions(pred_tr, "Transformer — prédictions sur le jeu de test")

## 7. Comparaison des trois modèles

In [ ]:
results = pd.DataFrame({
    "Modèle": ["RNN", "LSTM", "Transformer"],
    "Paramètres": [count_params(rnn_model), count_params(lstm_model), count_params(transformer_model)],
    "MSE test": [mse_rnn, mse_lstm, mse_tr],
    "MAE test": [mae_rnn, mae_lstm, mae_tr],
})
print(results.to_string(index=False))

plt.figure(figsize=(9,3.5))
plt.plot(y_test[:100].numpy(), "k", linewidth=2, label="Réel")
plt.plot(pred_rnn[:100].numpy(), "--", label="RNN")
plt.plot(pred_lstm[:100].numpy(), "--", label="LSTM")
plt.plot(pred_tr[:100].numpy(), "--", label="Transformer")
plt.legend()
plt.title("Comparaison des prédictions")
plt.grid(True, alpha=0.3)
plt.show()

## 8. Prévision récursive (multi-pas)

Jusqu’ici chaque prédiction utilise 10 **vraies** valeurs. Ici, le modèle reçoit seulement les 10 premières valeurs du test puis se **nourrit de ses propres prédictions** pour prédire les 100 pas suivants. Les petites erreurs s’accumulent : c’est un test beaucoup plus difficile.

In [ ]:
def forecast(model, seed_seq, steps=100):
    model.eval()
    seq = seed_seq.clone()                      # (10, 1)
    preds = []
    with torch.inference_mode():
        for _ in range(steps):
            nxt = model(seq.unsqueeze(0).to(device)).cpu()   # (1, 1)
            preds.append(nxt.item())
            seq = torch.cat([seq[1:], nxt], dim=0)           # on glisse la fenêtre
    return np.array(preds)

STEPS = 100
seed_seq = X_test[0]                                         # 10 premières valeurs du test
truth = series[split + SEQ_LEN : split + SEQ_LEN + STEPS].numpy()

plt.figure(figsize=(9,3.5))
plt.plot(truth, "k", linewidth=2, label="Réel")
for name, m in [("RNN", rnn_model), ("LSTM", lstm_model), ("Transformer", transformer_model)]:
    f = forecast(m, seed_seq, STEPS)
    print(f"{name:12s} | MAE sur {STEPS} pas = {np.abs(f - truth).mean():.4f}")
    plt.plot(f, "--", label=name)
plt.legend()
plt.title("Prévision récursive sur 100 pas")
plt.grid(True, alpha=0.3)
plt.show()

## 9. Transformer de type GPT — prédire la suite d’un texte

**GPT** (*Generative Pre-trained Transformer*) est un Transformer **décodeur** :
- il est **causal** : chaque mot ne voit que les mots **qui précèdent** (masque d’attention triangulaire) ;
- il est entraîné à prédire le **mot suivant** (*next-token prediction*) ;
- pour générer du texte, il prédit un mot, l’ajoute à l’entrée, puis recommence (**génération autorégressive**).

C’est exactement le même principe que notre prédiction « valeur suivante » sur `sin(x)`, mais sur des **tokens**.

Nous utilisons le modèle pré-entraîné `gpt2` de Hugging Face (téléchargé au premier lancement).

In [ ]:
from transformers import pipeline, set_seed, AutoTokenizer, AutoModelForCausalLM, AutoModelForMaskedLM

set_seed(42)
generator = pipeline("text-generation", model="gpt2", device=0 if torch.cuda.is_available() else -1)

prompt = "Artificial intelligence will change the world because"
outputs = generator(
    prompt,
    max_new_tokens=30,
    num_return_sequences=2,
    do_sample=True,
    top_k=50,
    temperature=0.8,
    pad_token_id=50256,
)

for i, o in enumerate(outputs, start=1):
    print(f"--- Génération {i} ---")
    print(o["generated_text"], "\n")

### Sous le capot : les probabilités du mot suivant

Le modèle produit, pour chaque position, un score (*logit*) pour **chaque mot du vocabulaire**. Une fonction `softmax` les transforme en probabilités. Regardons les 5 mots les plus probables après un début de phrase.

In [ ]:
tok_gpt = AutoTokenizer.from_pretrained("gpt2")
gpt = AutoModelForCausalLM.from_pretrained("gpt2")
gpt.eval()

text = "The capital of France is"
inputs = tok_gpt(text, return_tensors="pt")
print("Tokens :", tok_gpt.convert_ids_to_tokens(inputs["input_ids"][0]))

with torch.inference_mode():
    logits = gpt(**inputs).logits            # (1, nb_tokens, taille_vocabulaire)

probs = torch.softmax(logits[0, -1], dim=-1)  # distribution du MOT SUIVANT
top_p, top_i = probs.topk(5)

print("\nTop 5 des mots suivants :")
for p, i in zip(top_p, top_i):
    print(f"{tok_gpt.decode(i)!r:>12} -> {p.item():.3f}")

### Génération autorégressive « à la main » (choix glouton)

À chaque étape on prend le mot le plus probable (`argmax`), on l’ajoute à la séquence, et on recommence.

In [ ]:
ids = tok_gpt("The capital of France is", return_tensors="pt")["input_ids"]

for _ in range(10):
    with torch.inference_mode():
        logits = gpt(ids).logits
    next_id = logits[0, -1].argmax()
    ids = torch.cat([ids, next_id.view(1, 1)], dim=1)
    print(tok_gpt.decode(ids[0]))

## 10. Transformer de type BERT — prédire un mot caché

**BERT** (*Bidirectional Encoder Representations from Transformers*) est un Transformer **encodeur** :
- il est **bidirectionnel** : chaque mot voit les mots **avant et après** ;
- il est entraîné par **modélisation de langage masquée** (*Masked Language Modeling*) : on cache des mots avec `[MASK]` et le modèle doit les retrouver ;
- il sert surtout à **comprendre** un texte (classification, extraction, recherche…), pas à générer.

| | GPT | BERT |
|---|---|---|
| Bloc Transformer | décodeur | encodeur |
| Contexte vu | gauche uniquement (causal) | gauche **et** droite |
| Tâche d’entraînement | prédire le mot suivant | retrouver les mots masqués |
| Usage typique | génération de texte | compréhension de texte |

In [ ]:
fill_mask = pipeline("fill-mask", model="bert-base-uncased", device=0 if torch.cuda.is_available() else -1)

sentence = "The capital of France is [MASK]."
for r in fill_mask(sentence, top_k=5):
    print(f"{r['token_str']:>10} | score = {r['score']:.3f} | {r['sequence']}")

### Le contexte à droite compte

Ici le mot caché est déterminé par ce qui se trouve **après** lui (*to buy some bread*) : un modèle GPT, qui ne voit que la gauche, ne pourrait pas l’exploiter.

In [ ]:
sentence = "I went to the [MASK] to buy some bread."
for r in fill_mask(sentence, top_k=5):
    print(f"{r['token_str']:>10} | score = {r['score']:.3f}")

### Sous le capot : retrouver le `[MASK]` manuellement

In [ ]:
tok_bert = AutoTokenizer.from_pretrained("bert-base-uncased")
bert = AutoModelForMaskedLM.from_pretrained("bert-base-uncased")
bert.eval()

text = "Paris is the [MASK] of France."
inputs = tok_bert(text, return_tensors="pt")

mask_pos = (inputs["input_ids"][0] == tok_bert.mask_token_id).nonzero(as_tuple=True)[0]
print("Position du [MASK] :", mask_pos.item())

with torch.inference_mode():
    logits = bert(**inputs).logits            # (1, nb_tokens, taille_vocabulaire)

probs = torch.softmax(logits[0, mask_pos.item()], dim=-1)
top_p, top_i = probs.topk(5)

for p, i in zip(top_p, top_i):
    print(f"{tok_bert.decode(i):>10} -> {p.item():.3f}")

### Et en français ?

Le modèle `bert-base-multilingual-cased` a été entraîné sur une centaine de langues.

In [ ]:
fill_mask_fr = pipeline("fill-mask", model="bert-base-multilingual-cased", device=0 if torch.cuda.is_available() else -1)

for r in fill_mask_fr("Paris est la capitale de la [MASK].", top_k=5):
    print(f"{r['token_str']:>10} | score = {r['score']:.3f}")

### Synthèse de la première partie

- `RNN`, `LSTM` et `Transformer` apprennent tous à prédire la valeur suivante d’une sinusoïde ; la comparaison des MSE et de la prévision récursive montre leurs différences ;
- un `RNN` lit la séquence pas à pas, un `LSTM` améliore sa mémoire avec des portes, un `Transformer` regarde toute la séquence en parallèle avec l’attention ;
- **GPT** = Transformer décodeur causal → génère du texte mot après mot ;
- **BERT** = Transformer encodeur bidirectionnel → comprend le contexte et retrouve les mots masqués.

# Deuxième partie — Exercices

### Exercice 1 : Régression

Prédire le salaire selon l’ancienneté et le niveau d’expertise.

**Objectif :** construire un modèle de régression linéaire multiple pour prédire le salaire mensuel à partir de deux variables :
- Ancienneté : nombre d’années d’expérience ;
- Niveau d’expertise : note de 1 à 5.

Le modèle doit apprendre la relation :

\[
\text{Salaire} = w_1 \times \text{Ancienneté} + w_2 \times \text{Expertise} + b
\]

Les salaires sont exprimés en dirhams (DH). Les données sont fictives et servent uniquement à l’apprentissage.

**1. Générer les données**

On suppose que les salaires suivent approximativement la règle :

\[
\text{Salaire} = 4000 + 600 \times \text{Ancienneté} + 1500 \times \text{Expertise} + \text{Bruit}
\]

In [ ]:
torch.manual_seed(42)

# Nombre de salariés
n = 300

# Ancienneté : de 0 à 20 ans
anciennete = torch.randint(0, 21, (n, 1)).float()

# Expertise : de 1 à 5
expertise = torch.randint(1, 6, (n, 1)).float()

# Variations aléatoires des salaires
bruit = torch.randn(n, 1) * 500

# Salaire mensuel en DH
salaires = 4000 + 600 * anciennete + 1500 * expertise + bruit

# Matrice des entrées : (300, 2)
X_sal = torch.cat([anciennete, expertise], dim=1)

# Sortie : (300, 1)
y_sal = salaires

for i in range(5):
    print(
        f"Ancienneté : {X_sal[i, 0].item():.0f} ans | "
        f"Expertise : {X_sal[i, 1].item():.0f}/5 | "
        f"Salaire : {y_sal[i, 0].item():.2f} DH"
    )

**Travail demandé :**
- afficher les dimensions de `X` et de `y` ;
- calculer le salaire moyen ;
- tracer un nuage de points du salaire selon l’ancienneté, avec une couleur différente selon l’expertise.

In [ ]:
print("Shape X:", X_sal.shape)
print("Shape y:", y_sal.shape)
print(f"Salaire moyen : {y_sal.mean().item():.2f} DH")

In [ ]:
plt.figure(figsize=(6,4))
scatter = plt.scatter(X_sal[:, 0].numpy(), y_sal.numpy(), c=X_sal[:, 1].numpy(), s=18, cmap="viridis")
plt.colorbar(scatter, label="Expertise")
plt.title("Le salaire selon l’ancienneté")
plt.xlabel("Ancienneté (années)")
plt.ylabel("Salaire (DH)")
plt.grid(True, alpha=0.3)
plt.show()

**2. Séparer et préparer les données**

- 80 % des données pour l’entraînement, 20 % pour le test ;
- standardisation des entrées à partir des statistiques du jeu d’entraînement uniquement ;
- salaires exprimés en milliers de DH.

In [ ]:
indices = torch.randperm(n)
limite = int(0.8 * n)

train_indices = indices[:limite]
test_indices = indices[limite:]

X_train_sal = X_sal[train_indices]
X_test_sal = X_sal[test_indices]

y_train_sal = y_sal[train_indices] / 1000
y_test_sal = y_sal[test_indices] / 1000

# Statistiques calculées uniquement sur le jeu d'entraînement
moyenne = X_train_sal.mean(dim=0, keepdim=True)
ecart_type = X_train_sal.std(dim=0, keepdim=True)

X_train_norm = (X_train_sal - moyenne) / ecart_type
X_test_norm = (X_test_sal - moyenne) / ecart_type

print("Train :", X_train_norm.shape, "| Test :", X_test_norm.shape)

**3. Construire et entraîner le modèle**

In [ ]:
# Deux variables d'entrée et une sortie
model_sal = nn.Linear(2, 1)

# Erreur quadratique moyenne
criterion = nn.MSELoss()

optimizer = torch.optim.SGD(model_sal.parameters(), lr=0.05)

historique = []

model_sal.train()

for epoch in range(500):
    # 1. Réinitialiser les gradients
    optimizer.zero_grad()

    # 2. Prédire les salaires
    predictions = model_sal(X_train_norm)

    # 3. Calculer la perte
    loss = criterion(predictions, y_train_sal)

    # 4. Calculer les gradients
    loss.backward()

    # 5. Mettre à jour les paramètres
    optimizer.step()

    historique.append(loss.item())

    if (epoch + 1) % 100 == 0:
        print(f"Époque {epoch + 1} | Perte : {loss.item():.4f}")

In [ ]:
print("Weights :", model_sal.weight.data)
print("Bias    :", model_sal.bias.data)

In [ ]:
plt.plot(historique)
plt.xlabel("Époque")
plt.ylabel("MSE — salaires en milliers de DH")
plt.title("Évolution de la perte d'entraînement")
plt.grid()
plt.show()

**4. Évaluer les prédictions**

Interprétation : une MAE de 400 DH signifie que les prédictions s’écartent, en moyenne, de 400 DH des salaires réels.

In [ ]:
model_sal.eval()

with torch.no_grad():
    predictions_test = model_sal(X_test_norm)

    # Conversion en DH
    predictions_dh = predictions_test * 1000
    salaires_reels_dh = y_test_sal * 1000

    mae = torch.mean(torch.abs(predictions_dh - salaires_reels_dh))

print(f"Erreur absolue moyenne : {mae.item():.2f} DH")

**5. Prédire le salaire d’un nouveau salarié**

Une personne ayant 7 ans d’ancienneté et un niveau d’expertise de 4/5. D’après la règle de génération, le salaire sans bruit est `4000 + 600 × 7 + 1500 × 4 = 14200 DH`.

In [ ]:
nouveau_salarie = torch.tensor([[7.0, 4.0]])

# Appliquer la même normalisation qu'à l'entraînement
nouveau_salarie_norm = (nouveau_salarie - moyenne) / ecart_type

with torch.no_grad():
    salaire_predit = model_sal(nouveau_salarie_norm) * 1000

print(f"Salaire estimé : {salaire_predit.item():.2f} DH (valeur théorique sans bruit : 14200 DH)")

**Questions complémentaires**

*Retrouver les coefficients en unités d’origine.* Le modèle a appris `salaire_k = w · (x − μ)/σ + b` (en milliers de DH). En développant :

\[
\text{coef}_i = \frac{w_i}{\sigma_i} \times 1000, \qquad
\text{intercept} = \left(b - \sum_i \frac{w_i \mu_i}{\sigma_i}\right) \times 1000
\]

In [ ]:
w = model_sal.weight.data.squeeze()      # (2,)
b = model_sal.bias.data.item()
mu = moyenne.squeeze()
sd = ecart_type.squeeze()

coef_anciennete = (w[0] / sd[0]).item() * 1000
coef_expertise = (w[1] / sd[1]).item() * 1000
intercept = (b - (w * mu / sd).sum().item()) * 1000

print(f"Ancienneté : {coef_anciennete:.1f} DH / an      (théorique : 600)")
print(f"Expertise  : {coef_expertise:.1f} DH / niveau  (théorique : 1500)")
print(f"Intercept  : {intercept:.1f} DH                 (théorique : 4000)")

*Effet du bruit sur la précision.* On réentraîne le même modèle avec des niveaux de bruit croissants.

In [ ]:
def entrainer_salaires(bruit_std, seed=0):
    torch.manual_seed(seed)
    n = 300
    anc = torch.randint(0, 21, (n, 1)).float()
    exp = torch.randint(1, 6, (n, 1)).float()
    sal = 4000 + 600 * anc + 1500 * exp + torch.randn(n, 1) * bruit_std

    Xs = torch.cat([anc, exp], dim=1)
    ys = sal / 1000

    idx = torch.randperm(n)
    lim = int(0.8 * n)
    tr, te = idx[:lim], idx[lim:]

    mu_ = Xs[tr].mean(dim=0, keepdim=True)
    sd_ = Xs[tr].std(dim=0, keepdim=True)
    Xtr, Xte = (Xs[tr] - mu_) / sd_, (Xs[te] - mu_) / sd_

    m = nn.Linear(2, 1)
    opt = torch.optim.SGD(m.parameters(), lr=0.05)
    for _ in range(500):
        opt.zero_grad()
        l = nn.MSELoss()(m(Xtr), ys[tr])
        l.backward()
        opt.step()

    with torch.no_grad():
        return ((m(Xte) - ys[te]).abs().mean() * 1000).item()

for s in [0, 250, 500, 1000, 2000]:
    print(f"bruit = {s:5d} DH -> MAE test = {entrainer_salaires(s):8.2f} DH")

**Réponses :**

- **Effet d’une année d’ancienneté supplémentaire (à expertise constante)** : c’est le coefficient `coef_anciennete`, soit environ **+600 DH** de salaire mensuel.
- **Pourquoi la perte n’atteint pas 0** : le bruit (écart-type 500 DH) représente des variations individuelles qu’aucune des deux variables n’explique. Même un modèle parfait ne peut pas les prédire ; la perte minimale tend vers la variance du bruit (0,25 en milliers de DH au carré).
- **Fiabilité à 40 ans d’ancienneté** : non. Le modèle n’a vu que des anciennetés de 0 à 20 ans ; au-delà, c’est de l’**extrapolation**. La relation linéaire n’y est pas garantie (dans la réalité, les salaires plafonnent souvent).
- **Effet d’une augmentation du bruit** : la MAE augmente à peu près proportionnellement au bruit (voir le tableau ci-dessus) et les coefficients estimés deviennent plus instables.
- **Remarque** : traiter l’expertise comme une note numérique suppose que chaque passage d’un niveau au suivant a le même effet sur le salaire. Cette hypothèse simplifie l’exercice, mais doit être vérifiée avec des données réelles.

### Exercice 2 : Classifier les chiffres MNIST

**Objectif :** utiliser des mini-lots (*mini-batches*) et évaluer un modèle sur des données de test.

**Chargement des données**

In [ ]:
from torchvision import datasets, transforms

transform = transforms.ToTensor()

train_dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

train_loader_mnist = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader_mnist = DataLoader(test_dataset, batch_size=256, shuffle=False)

print("Images d'entraînement :", len(train_dataset))
print("Images de test        :", len(test_dataset))

img, label = train_dataset[0]
print("Dimensions d'une image :", img.shape, "| étiquette :", label)

In [ ]:
fig, axes = plt.subplots(1, 8, figsize=(12, 2))
for ax, i in zip(axes, range(8)):
    img, label = train_dataset[i]
    ax.imshow(img.squeeze(), cmap="gray")
    ax.set_title(label)
    ax.axis("off")
plt.show()

In [ ]:
model_mnist = nn.Sequential(
    nn.Flatten(),
    nn.Linear(28 * 28, 128),
    nn.ReLU(),
    nn.Linear(128, 10)
).to(device)

criterion_mnist = nn.CrossEntropyLoss()
optimizer_mnist = torch.optim.Adam(model_mnist.parameters(), lr=0.001)
print(model_mnist)

**Travail demandé :**
- entraîner le modèle pendant 5 époques ;
- utiliser `model.train()` pendant l’entraînement ;
- évaluer le modèle sur le jeu de test avec `model.eval()`.

Chaque époque parcourt tous les mini-lots de 64 images : forward → loss → `zero_grad()` → `backward()` → `step()`.

In [ ]:
epochs_mnist = 5
historique_mnist = []

for epoch in range(epochs_mnist):
    model_mnist.train()
    running = 0.0

    for images, labels in train_loader_mnist:
        images, labels = images.to(device), labels.to(device)

        optimizer_mnist.zero_grad()
        outputs = model_mnist(images)
        loss = criterion_mnist(outputs, labels)
        loss.backward()
        optimizer_mnist.step()

        running += loss.item() * images.size(0)

    epoch_loss = running / len(train_dataset)
    historique_mnist.append(epoch_loss)
    print(f"Époque {epoch + 1}/{epochs_mnist} | Perte moyenne : {epoch_loss:.4f}")

In [ ]:
plt.plot(range(1, epochs_mnist + 1), historique_mnist, marker="o")
plt.xlabel("Époque")
plt.ylabel("Perte moyenne")
plt.title("MNIST — évolution de la perte")
plt.grid(True, alpha=0.3)
plt.show()

**Évaluation sur le jeu de test**

In [ ]:
model_mnist.eval()

correct, total = 0, 0
all_preds, all_labels = [], []

with torch.no_grad():
    for images, labels in test_loader_mnist:
        images, labels = images.to(device), labels.to(device)
        preds = model_mnist(images).argmax(dim=1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)
        all_preds.append(preds.cpu())
        all_labels.append(labels.cpu())

all_preds = torch.cat(all_preds)
all_labels = torch.cat(all_labels)

print(f"Accuracy sur le test : {correct / total:.2%}  ({correct}/{total})")

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_predictions(all_labels.numpy(), all_preds.numpy(), cmap="Blues")
plt.title("MNIST — matrice de confusion")
plt.show()

In [ ]:
# Quelques prédictions
images, labels = next(iter(test_loader_mnist))
with torch.no_grad():
    preds = model_mnist(images.to(device)).argmax(dim=1).cpu()

fig, axes = plt.subplots(1, 8, figsize=(12, 2.4))
for ax, i in zip(axes, range(8)):
    ax.imshow(images[i].squeeze(), cmap="gray")
    ax.set_title(f"préd={preds[i].item()}\nréel={labels[i].item()}",
                 color="green" if preds[i] == labels[i] else "red", fontsize=9)
    ax.axis("off")
plt.show()

### Exercice 3 : Classification des fleurs Iris 🌸

**Objectif :** entraîner un réseau de neurones pour prédire l’espèce d’une fleur à partir de quatre mesures :
- longueur et largeur des sépales ;
- longueur et largeur des pétales.

Le jeu de données Iris contient 150 fleurs réparties en trois classes : *setosa*, *versicolor* et *virginica*.

**1. Préparation des données**

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

torch.manual_seed(42)

# Charger les données
iris = load_iris()
X_iris, y_iris = iris.data, iris.target

# Séparer les données : 80 % entraînement, 20 % test
X_train_i, X_test_i, y_train_i, y_test_i = train_test_split(
    X_iris, y_iris, test_size=0.2, random_state=42, stratify=y_iris
)

# Ajuster la normalisation uniquement sur l'entraînement
scaler = StandardScaler()
X_train_i = scaler.fit_transform(X_train_i)
X_test_i = scaler.transform(X_test_i)

# Convertir en tenseurs
X_train_i = torch.tensor(X_train_i, dtype=torch.float32)
X_test_i = torch.tensor(X_test_i, dtype=torch.float32)
y_train_i = torch.tensor(y_train_i, dtype=torch.long)
y_test_i = torch.tensor(y_test_i, dtype=torch.long)

print("Dimensions :", X_train_i.shape, y_train_i.shape)
print("Colonnes   :", iris.feature_names)
print("Classes    :", list(iris.target_names))

**Réponses aux questions :**

- **Les quatre colonnes de `X_train`** : longueur du sépale, largeur du sépale, longueur du pétale, largeur du pétale (en cm), ici standardisées (moyenne 0, écart-type 1).
- **Pourquoi `torch.long` pour les étiquettes ?** `CrossEntropyLoss` attend des **indices de classe** (entiers 64 bits : 0, 1 ou 2), pas des nombres décimaux.
- **Pourquoi ne pas ajuster le `StandardScaler` sur le test ?** Cela ferait fuiter de l’information du jeu de test vers l’entraînement (*data leakage*) et fausserait l’évaluation. On utilise `fit_transform` sur le train, puis seulement `transform` sur le test.

**2. Construire le réseau**

Attention : pas de `Softmax` en sortie. `CrossEntropyLoss` attend directement les scores bruts, appelés *logits*.

In [ ]:
model_iris = nn.Sequential(
    nn.Linear(4, 16),
    nn.ReLU(),
    nn.Linear(16, 3)
)

criterion_iris = nn.CrossEntropyLoss()
optimizer_iris = torch.optim.Adam(model_iris.parameters(), lr=0.01)
print(model_iris)

**3. Entraîner le modèle**

Chaque itération utilise toutes les données d’entraînement.

In [ ]:
epochs = 200
losses_iris = []

model_iris.train()

for epoch in range(epochs):
    # remettre les gradients à zéro
    optimizer_iris.zero_grad()

    # calculer les scores prédits
    logits = model_iris(X_train_i)

    # calculer la perte
    loss = criterion_iris(logits, y_train_i)

    # calculer les gradients
    loss.backward()

    # mettre à jour les paramètres
    optimizer_iris.step()

    losses_iris.append(loss.item())

    if (epoch + 1) % 20 == 0:
        print(f"Époque {epoch + 1:3d} | Perte : {loss.item():.4f}")

**4. Évaluer le classifieur**

L’accuracy est la proportion de fleurs correctement classées.

In [ ]:
model_iris.eval()

with torch.no_grad():
    logits = model_iris(X_test_i)

    # classe au score maximal pour chaque fleur
    predictions_iris = logits.argmax(dim=1)

    # proportion de prédictions correctes
    accuracy = (predictions_iris == y_test_i).float().mean()

print(f"Accuracy sur le test : {accuracy.item():.1%}")

**5. Autres questions**

- courbe de perte ;
- matrice de confusion ;
- comparaison avec un modèle sans couche cachée `nn.Linear(4, 3)` ;
- prédiction de l’espèce d’une nouvelle fleur `[5.1, 3.5, 1.4, 0.2]`.

In [ ]:
plt.plot(losses_iris)
plt.xlabel("Époque")
plt.ylabel("Perte")
plt.title("Iris — évolution de la perte d'entraînement")
plt.grid()
plt.show()

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_predictions(
    y_test_i.numpy(), predictions_iris.numpy(),
    display_labels=iris.target_names, cmap="Blues"
)
plt.title("Iris — matrice de confusion")
plt.show()

**Comparaison : réseau avec couche cachée vs modèle linéaire**

In [ ]:
torch.manual_seed(42)
model_simple = nn.Linear(4, 3)

criterion_simple = nn.CrossEntropyLoss()
optimizer_simple = torch.optim.Adam(model_simple.parameters(), lr=0.01)

model_simple.train()
for epoch in range(200):
    optimizer_simple.zero_grad()
    loss = criterion_simple(model_simple(X_train_i), y_train_i)
    loss.backward()
    optimizer_simple.step()

model_simple.eval()
with torch.no_grad():
    pred_simple = model_simple(X_test_i).argmax(dim=1)

acc_simple = (pred_simple == y_test_i).float().mean()

print(f"Réseau (4-16-3)       : accuracy = {accuracy.item():.1%}")
print(f"Linéaire (4-3)        : accuracy = {acc_simple.item():.1%}")

Sur Iris, les classes sont presque linéairement séparables : le modèle linéaire donne souvent un résultat très proche du réseau avec couche cachée. La couche cachée devient vraiment utile quand la frontière entre classes est non linéaire.

**Prédire l’espèce d’une nouvelle fleur**

La nouvelle fleur doit être normalisée avec **le même scaler** que les données d’entraînement.

In [ ]:
nouvelle_fleur_scaled = scaler.transform([[5.1, 3.5, 1.4, 0.2]])
nouvelle_fleur_tensor = torch.tensor(nouvelle_fleur_scaled, dtype=torch.float32)

model_iris.eval()
with torch.no_grad():
    outputs = model_iris(nouvelle_fleur_tensor)
    probas = torch.softmax(outputs, dim=1)[0]
    prediction = outputs.argmax(dim=1).item()

especes = ["Setosa", "Versicolor", "Virginica"]

print("Espèce prédite :", especes[prediction])
for e, p in zip(especes, probas):
    print(f"  {e:10s} : {p.item():.1%}")

# Synthèse

**Première partie**
1. une série temporelle se transforme en exemples `(séquence de 10 valeurs → valeur suivante)` par fenêtre glissante ;
2. `RNN`, `LSTM` et `Transformer` se branchent sur la même boucle d’apprentissage PyTorch ;
3. un RNN lit pas à pas, un LSTM ajoute des portes de mémoire, un Transformer utilise l’attention sur toute la séquence ;
4. GPT (décodeur, causal) génère du texte ; BERT (encodeur, bidirectionnel) retrouve des mots masqués.

**Deuxième partie**
1. régression linéaire multiple : standardiser les entrées avec les statistiques du train, puis retrouver les coefficients en unités d’origine ;
2. MNIST : mini-lots avec `DataLoader`, `model.train()` pour apprendre, `model.eval()` + `torch.no_grad()` pour tester ;
3. Iris : `CrossEntropyLoss` sur des logits, étiquettes en `torch.long`, scaler ajusté sur le train uniquement.